<a href="https://colab.research.google.com/github/Lathika-M732/-24ADI003_24BAD062_EX-4/blob/main/EX_4_DL_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# EXPT NO: 4
# TRANSFER LEARNING USING PRE-TRAINED VISION MODELS
# FOR IMAGE RECOGNITION
# ============================================================

STUDENT_NAME = "Lathika M."
ROLL_NO = "24BAD062"

print("=" * 70)
print("TRANSFER LEARNING USING PRE-TRAINED VISION MODELS")
print("=" * 70)
print("Student Name : ", STUDENT_NAME)
print("Roll Number  : ", ROLL_NO)
print("Experiment No: 4")
print("=" * 70)


# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing import image_dataset_from_directory

print("\nLibraries imported successfully.")
print("TensorFlow Version:", tf.__version__)


# ============================================================
# STEP 2: CHECK GPU
# ============================================================

print("\n" + "=" * 70)
print("CHECKING GPU")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU is available.")
    print("GPU Details:", gpus)
else:
    print("GPU is not available.")
    print("CPU will be used.")

print("=" * 70)


# ============================================================
# STEP 3: UPLOAD DATASET
# ============================================================

from google.colab import files

print("\n" + "=" * 70)
print("UPLOAD DATASET")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)
print("Please upload: Intel Image Dataset.zip")
print("=" * 70)

uploaded = files.upload()

uploaded_files = list(uploaded.keys())

print("\nUploaded file(s):")
for filename in uploaded_files:
    print(filename)


# ============================================================
# STEP 4: FIND ZIP FILE AUTOMATICALLY
# ============================================================

zip_file = None

for filename in uploaded_files:
    if filename.lower().endswith(".zip"):
        zip_file = "/content/" + filename
        break

if zip_file is None:
    raise FileNotFoundError(
        "No ZIP file was uploaded. Please upload the Intel Image Dataset ZIP file."
    )

print("\nZIP file selected:")
print(zip_file)


# ============================================================
# STEP 5: EXTRACT DATASET
# ============================================================

EXTRACT_PATH = "/content/intel_dataset"

print("\n" + "=" * 70)
print("EXTRACTING DATASET")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")
print("Extraction path:", EXTRACT_PATH)


# ============================================================
# STEP 6: FIND DATASET FOLDER
# ============================================================

print("\n" + "=" * 70)
print("SEARCHING FOR DATASET FOLDER")
print("=" * 70)

expected_classes = {
    "buildings",
    "forest",
    "glacier",
    "mountain",
    "sea",
    "street"
}

DATASET_PATH = None

for root, dirs, files_list in os.walk(EXTRACT_PATH):

    current_folders = set(folder.lower() for folder in dirs)

    if expected_classes.issubset(current_folders):
        DATASET_PATH = root
        break

if DATASET_PATH is None:

    print("Automatic detection failed.")
    print("Folders found in extracted dataset:")

    for root, dirs, files_list in os.walk(EXTRACT_PATH):
        print(root)
        print(dirs)

    raise FileNotFoundError(
        "Could not locate the six Intel image classes."
    )

print("Dataset folder found:")
print(DATASET_PATH)


# ============================================================
# STEP 7: DISPLAY DATASET CLASSES
# ============================================================

print("\n" + "=" * 70)
print("DATASET CLASSES")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

class_names = sorted([
    folder
    for folder in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, folder))
])

print("\nNumber of Classes:", len(class_names))

for i, class_name in enumerate(class_names):
    print(i, ":", class_name)


# ============================================================
# STEP 8: COUNT IMAGES IN EACH CLASS
# ============================================================

print("\n" + "=" * 70)
print("IMAGE COUNT")
print("=" * 70)

total_images = 0

for class_name in class_names:

    class_path = os.path.join(DATASET_PATH, class_name)

    image_count = 0

    for filename in os.listdir(class_path):

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".gif")
        ):
            image_count += 1

    total_images += image_count

    print(f"{class_name:15s}: {image_count} images")

print("-" * 50)
print("Total Images:", total_images)


# ============================================================
# STEP 9: SET DATASET PARAMETERS
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("\n" + "=" * 70)
print("DATASET PARAMETERS")
print("=" * 70)

print("Image Size :", IMG_SIZE)
print("Batch Size :", BATCH_SIZE)
print("Random Seed:", SEED)


# ============================================================
# STEP 10: LOAD TRAINING DATASET
# ============================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATASET")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

train_ds = image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

print("\nTraining dataset loaded successfully.")


# ============================================================
# STEP 11: LOAD VALIDATION + TEST DATASET
# ============================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION AND TEST DATASET")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

remaining_ds = image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

remaining_batches = tf.data.experimental.cardinality(
    remaining_ds
).numpy()

val_batches = remaining_batches // 2

val_ds = remaining_ds.take(val_batches)
test_ds = remaining_ds.skip(val_batches)

print("\nDataset split completed.")
print("Training   : approximately 70%")
print("Validation : approximately 15%")
print("Testing    : approximately 15%")


# ============================================================
# STEP 12: OPTIMIZE DATA PIPELINE
# ============================================================

print("\n" + "=" * 70)
print("OPTIMIZING DATA PIPELINE")
print("=" * 70)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("Dataset pipeline optimized successfully.")


# ============================================================
# STEP 13: DISPLAY SAMPLE IMAGES
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE IMAGES")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):

    for i in range(min(12, len(images))):

        ax = plt.subplot(3, 4, i + 1)

        plt.imshow(
            images[i].numpy().astype("uint8")
        )

        plt.title(
            class_names[labels[i].numpy()]
        )

        plt.axis("off")

plt.tight_layout()
plt.show()


# ============================================================
# STEP 14: LOAD PRE-TRAINED RESNET50
# ============================================================

print("\n" + "=" * 70)
print("LOADING PRE-TRAINED RESNET50")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("ResNet50 loaded successfully.")
print("Pre-trained weights: ImageNet")
print("Classification layer removed.")


# ============================================================
# STEP 15: FREEZE RESNET50 LAYERS
# ============================================================

print("\n" + "=" * 70)
print("FREEZING FEATURE EXTRACTION LAYERS")
print("=" * 70)

base_model.trainable = False

print("All ResNet50 feature extraction layers are frozen.")
print("Only the new classification layers will be trained.")


# ============================================================
# STEP 16: DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


# ============================================================
# STEP 17: BUILD TRANSFER LEARNING MODEL
# ============================================================

print("\n" + "=" * 70)
print("BUILDING TRANSFER LEARNING MODEL")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

inputs = tf.keras.Input(
    shape=(224, 224, 3)
)

x = data_augmentation(inputs)

x = preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)

print("Transfer learning model created successfully.")
print("Output Classes:", len(class_names))


# ============================================================
# STEP 18: DISPLAY MODEL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

model.summary()


# ============================================================
# STEP 19: COMPILE MODEL
# ============================================================

print("\n" + "=" * 70)
print("COMPILING MODEL")
print("=" * 70)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")
print("Optimizer : Adam")
print("Learning Rate: 0.0001")
print("Loss      : Sparse Categorical Crossentropy")
print("Metric    : Accuracy")


# ============================================================
# STEP 20: TRAIN MODEL
# ============================================================

print("\n" + "=" * 70)
print("TRAINING TRANSFER LEARNING MODEL")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

print("\nTraining completed successfully.")


# ============================================================
# STEP 21: DISPLAY TRAINING PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("TRAINING PERFORMANCE")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

train_accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

for epoch in range(len(train_accuracy)):

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Training Accuracy: {train_accuracy[epoch]:.4f} | "
        f"Validation Accuracy: {val_accuracy[epoch]:.4f} | "
        f"Training Loss: {train_loss[epoch]:.4f} | "
        f"Validation Loss: {val_loss[epoch]:.4f}"
    )


# ============================================================
# STEP 22: PLOT ACCURACY VS EPOCH
# ============================================================

print("\n" + "=" * 70)
print("ACCURACY VS EPOCH")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(train_accuracy) + 1),
    train_accuracy,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    range(1, len(val_accuracy) + 1),
    val_accuracy,
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(
    "ResNet50 Transfer Learning - Accuracy vs Epoch"
)
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# STEP 23: PLOT LOSS VS EPOCH
# ============================================================

print("\n" + "=" * 70)
print("LOSS VS EPOCH")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(train_loss) + 1),
    train_loss,
    marker="o",
    label="Training Loss"
)

plt.plot(
    range(1, len(val_loss) + 1),
    val_loss,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "ResNet50 Transfer Learning - Loss vs Epoch"
)
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# STEP 24: TEST MODEL
# ============================================================

print("\n" + "=" * 70)
print("TESTING TRANSFER LEARNING MODEL")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

test_loss, test_accuracy = model.evaluate(
    test_ds
)

print("\nTest Results")
print("-" * 50)
print("Test Loss     :", round(test_loss, 4))
print("Test Accuracy :", round(test_accuracy * 100, 2), "%")
print("-" * 50)


# ============================================================
# STEP 25: FINAL PERFORMANCE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL PERFORMANCE SUMMARY")
print("=" * 70)

print("Student Name             :", STUDENT_NAME)
print("Roll Number              :", ROLL_NO)
print("Experiment Number        : 4")
print("Pre-trained Model        : ResNet50")
print("Dataset                  : Intel Image Classification")
print("Number of Classes        :", len(class_names))
print("Number of Epochs         :", EPOCHS)

print(
    "Final Training Accuracy  :",
    round(train_accuracy[-1] * 100, 2),
    "%"
)

print(
    "Final Validation Accuracy:",
    round(val_accuracy[-1] * 100, 2),
    "%"
)

print(
    "Final Training Loss      :",
    round(train_loss[-1], 4)
)

print(
    "Final Validation Loss    :",
    round(val_loss[-1], 4)
)

print(
    "Test Accuracy            :",
    round(test_accuracy * 100, 2),
    "%"
)


# ============================================================
# STEP 26: INSTALL HUGGING FACE
# ============================================================

print("\n" + "=" * 70)
print("INSTALLING HUGGING FACE TRANSFORMERS")
print("=" * 70)

!pip install -q transformers torch torchvision

print("Hugging Face Transformers installed successfully.")


# ============================================================
# STEP 27: LOAD HUGGING FACE PRE-TRAINED MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING HUGGING FACE VISION MODEL")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

from transformers import pipeline

hf_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

print("Hugging Face model loaded successfully.")
print("Model: google/vit-base-patch16-224")


# ============================================================
# STEP 28: SELECT SAMPLE IMAGE
# ============================================================

print("\n" + "=" * 70)
print("SELECTING SAMPLE IMAGE")
print("=" * 70)

sample_image_path = None
actual_class = None

for class_name in class_names:

    class_path = os.path.join(
        DATASET_PATH,
        class_name
    )

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp")
        ):

            sample_image_path = os.path.join(
                class_path,
                file_name
            )

            actual_class = class_name

            break

    if sample_image_path is not None:
        break

print("Sample Image :", sample_image_path)
print("Actual Class :", actual_class)


# ============================================================
# STEP 29: DISPLAY SAMPLE IMAGE
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE IMAGE")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

sample_img = image.load_img(
    sample_image_path
)

plt.figure(figsize=(6, 6))

plt.imshow(sample_img)

plt.title(
    "Actual Class: " + actual_class
)

plt.axis("off")

plt.show()


# ============================================================
# STEP 30: HUGGING FACE IMAGE CLASSIFICATION
# ============================================================

print("\n" + "=" * 70)
print("HUGGING FACE IMAGE CLASSIFICATION")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

hf_results = hf_classifier(
    sample_image_path
)

print("\nTop Hugging Face Predictions")
print("-" * 50)

for result in hf_results[:5]:

    print(
        "Label:",
        result["label"],
        "| Score:",
        round(result["score"], 4)
    )


# ============================================================
# STEP 31: RESNET50 PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("RESNET50 IMAGE PREDICTION")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

img = image.load_img(
    sample_image_path,
    target_size=IMG_SIZE
)

img_array = image.img_to_array(
    img
)

img_array = np.expand_dims(
    img_array,
    axis=0
)

predictions = model.predict(
    img_array,
    verbose=0
)

predicted_index = np.argmax(
    predictions[0]
)

predicted_class = class_names[
    predicted_index
]

confidence = (
    predictions[0][predicted_index] * 100
)

print("Actual Class       :", actual_class)
print("ResNet50 Prediction :", predicted_class)
print("Confidence          :", round(confidence, 2), "%")


# ============================================================
# STEP 32: MODEL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print("Student Name :", STUDENT_NAME)
print("Roll Number  :", ROLL_NO)

print("\nActual Dataset Class:")
print(actual_class)

print("\nResNet50 Transfer Learning:")
print("Prediction :", predicted_class)
print("Confidence :", round(confidence, 2), "%")

print("\nHugging Face Vision Transformer:")
print("Top Prediction :", hf_results[0]["label"])
print(
    "Confidence     :",
    round(hf_results[0]["score"] * 100, 2),
    "%"
)


# ============================================================
# STEP 33: OBSERVATION
# ============================================================

print("\n" + "=" * 70)
print("OBSERVATION")
print("=" * 70)

if predicted_class.lower() == actual_class.lower():

    print(
        "ResNet50 correctly classified the sample image."
    )

else:

    print(
        "ResNet50 prediction differs from the dataset label."
    )

print(
    "Hugging Face Vision Transformer generated "
    "predictions using pre-trained ImageNet knowledge."
)

print(
    "Transfer learning reduces training time and "
    "computational requirements compared with training "
    "a CNN completely from scratch."
)


# ============================================================
# STEP 34: SAVE TRAINED MODEL
# ============================================================

print("\n" + "=" * 70)
print("SAVING TRAINED MODEL")
print("=" * 70)

MODEL_PATH = "/content/resnet50_transfer_learning.keras"

model.save(
    MODEL_PATH
)

print("Model saved successfully.")
print("Model Path:", MODEL_PATH)


# ============================================================
# STEP 35: FINAL OUTPUT
# ============================================================

print("\n\n")
print("=" * 75)
print("                    EXPERIMENT COMPLETED")
print("=" * 75)

print("Experiment No       : 4")
print("Title               : Transfer Learning Using")
print("                      Pre-trained Vision Models")
print("                      for Image Recognition")

print("-" * 75)

print("Student Name        :", STUDENT_NAME)
print("Roll Number         :", ROLL_NO)

print("-" * 75)

print("Dataset             : Intel Image Classification Dataset")
print("Pre-trained Model   : ResNet50")
print("Number of Classes   :", len(class_names))
print("Epochs              :", EPOCHS)

print(
    "Test Accuracy       :",
    round(test_accuracy * 100, 2),
    "%"
)

print(
    "Hugging Face Model  : google/vit-base-patch16-224"
)

print("-" * 75)

print(
    "Transfer learning implementation "
    "completed successfully."
)

print("=" * 75)

TRANSFER LEARNING USING PRE-TRAINED VISION MODELS
Student Name :  Lathika M.
Roll Number  :  24BAD062
Experiment No: 4

Libraries imported successfully.
TensorFlow Version: 2.20.0

CHECKING GPU
Student Name : Lathika M.
Roll Number  : 24BAD062
GPU is not available.
CPU will be used.

UPLOAD DATASET
Student Name : Lathika M.
Roll Number  : 24BAD062
Please upload: Intel Image Dataset.zip


TypeError: 'NoneType' object is not subscriptable